[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarioSigal/TP_Rompecabezas/blob/main/TP_NIVEL_6.ipynb)

# 🧩 Trabajo Práctico: Desafío Integrador — Nivel 6
**Procesamiento de Imágenes (PDI) — Resolución Automatizada de Rompecabezas**

**Grupo:** ____________________

**Integrantes:**
- ____________________
- ____________________
- ____________________

---

## 🎯 Objetivo
En este desafío final integrador deberán diseñar e implementar su propio pipeline de resolución automática (`mi_pipeline`) capaz de resolver un conjunto de **30 rompecabezas** afectados por múltiples degradaciones combinadas (ruido espacial, alteraciones cromáticas, tramas sinusoidales periódicas en Fourier, encastres curvos de rompecabezas y rotaciones ortogonales/inclinaciones).


---
## ⚙️ Configuración del Entorno de Ejecución

Esta celda configura automáticamente las rutas necesarias y descarga el repositorio si se ejecuta en Google Colab.


In [ ]:
# Configuración de entorno para Google Colab y ejecución local
import os, sys
from pathlib import Path

if 'google.colab' in str(get_ipython()):
    print('--> Entorno detectado: Google Colab')
    if not os.path.exists('repo_tp') and not os.path.exists('core'):
        !git clone https://github.com/MarioSigal/TP_Rompecabezas.git repo_tp
        %cd repo_tp
    else:
        if os.path.exists('repo_tp'):
            %cd repo_tp
        !git pull origin main
    sys.path.insert(0, os.getcwd())
else:
    print('--> Entorno detectado: Local')
    raiz = Path.cwd()
    if (raiz / 'TP_FINAL_ALUMNOS' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_FINAL_ALUMNOS'))
    elif (raiz / 'core').exists():
        sys.path.insert(0, str(raiz))
    elif (raiz / 'TP_Rompecabezas' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_Rompecabezas'))

import numpy as np
import cv2
import matplotlib.pyplot as plt

# Importar funciones y herramientas del núcleo del TP
from core import (
    cargar_imagen,
    guardar_imagen,
    preparar_imagen_base,
    crear_rompecabezas_nivel,
    generar_desafio_30,
    reconstruir_rompecabezas,
    reconstruir_desde_afinidades,
    construir_matrices_afinidad,
    compatibilidad_baseline,
    segmentar_borde_en_4,
    binarize_piece,
    extract_external_contour,
    estimar_orientacion_fourier,
    enderezar_pieza,
    generar_reporte_completo,
    imprimir_reporte,
)

from utils import (
    mostrar_piezas_desordenadas,
    mostrar_comparacion_imagen,
    mostrar_espectro_fourier,
    mostrar_reconstruccion,
    crear_animacion,
)

print('✅ ¡Módulos del TP cargados con éxito!')


---
## 🛠️ 1. Pipeline de Reconstrucción (Alumnos)

Implementen en la siguiente función su pipeline completo para resolver un rompecabezas del Nivel 6. Pueden definir las funciones y clases que consideren necesarias.


In [ ]:
def mi_pipeline(rompecabezas):
    """
    Pipeline de reconstrucción desarrollado por los alumnos para el Nivel 6.
    
    Parámetros:
    -----------
    rompecabezas : Rompecabezas
        Instancia de la clase Rompecabezas con las piezas desordenadas y degradadas:
        - rompecabezas.piezas: lista de arrays (H, W, 3) con las imágenes de cada pieza.
        - rompecabezas.cantidad_filas: cantidad de filas de la grilla.
        - rompecabezas.cantidad_columnas: cantidad de columnas de la grilla.
        - rompecabezas.cantidad_piezas: total de piezas (filas * columnas).
        - rompecabezas.metadatos: diccionario con metadatos del caso.
        
    Retorna:
    --------
    np.ndarray o tuple:
        - grilla_propuesta: np.ndarray de forma (filas, columnas) con los índices enteros
          de las piezas asignadas a cada posición de la grilla.
        - (Opcional) matrices_afinidad: dict con las matrices de afinidad calculadas.
    """
    # =========================================================================
    # TODO: Implementar aquí el pipeline completo del grupo
    # =========================================================================
    
    raise NotImplementedError("Deben implementar su pipeline de resolución en esta función.")


---
## 🎲 2. Generación del Desafío Integrador (30 Rompecabezas)

Ingresen la semilla asignada a su grupo. La función `generar_desafio_30` creará los 30 rompecabezas del dataset con las configuraciones y combinaciones del desafío.


In [ ]:
# =========================================================================
# CONFIGURACIÓN DEL DESAFÍO
# Ingrese la semilla única asignada a su grupo:
# =========================================================================
SEMILLA_GRUPO = 100

# Generación de los 30 rompecabezas del desafío integrador
puzzles_desafio = generar_desafio_30(semilla=SEMILLA_GRUPO)
print(f'\n✅ Se generaron exitosamente los {len(puzzles_desafio)} rompecabezas del desafío.')


---
## 📊 3. Evaluación y Cálculo de Métricas

Esta celda ejecuta su pipeline sobre cada uno de los 30 rompecabezas del desafío y registra las métricas de precisión obtenidas.


In [ ]:
reportes = []

print(f'🧩 Iniciando evaluación de los {len(puzzles_desafio)} rompecabezas con su pipeline...\n')

for i, puzzle in enumerate(puzzles_desafio):
    nombre_img = puzzle.metadatos.get('archivo_origen', f'puzzle_{i+1}')
    print(f'Resolviendo rompecabezas {i + 1:2d}/{len(puzzles_desafio)}: [{nombre_img}]...', end=' ')
    
    # Ejecución del pipeline de los alumnos
    salida = mi_pipeline(puzzle)
    
    if isinstance(salida, tuple):
        grilla = salida[0]
        matrices = salida[1] if len(salida) > 1 else None
    elif isinstance(salida, dict):
        grilla = salida.get('grilla', salida.get('grilla_propuesta'))
        matrices = salida.get('matrices_afinidad', None)
    else:
        grilla = salida
        matrices = None
        
    rep = generar_reporte_completo(puzzle, matrices_afinidad=matrices, grilla_propuesta=grilla)
    reportes.append(rep)
    
    vec = rep.get('precision_vecindad', 0.0) * 100.0
    dir_acc = rep.get('precision_directa', 0.0) * 100.0
    print(f'Vecindad: {vec:5.1f}% | Directa: {dir_acc:5.1f}%')

print('\n✅ Evaluación completada en los 30 rompecabezas.')


---
## 🏆 4. Resultados y Métricas Globales Promedio


In [ ]:
def calcular_reporte_promedio(reportes: list) -> dict:
    if not reportes:
        return {}
    suma = {}
    conteo = {}
    for r in reportes:
        for k, v in r.items():
            if isinstance(v, (int, float)):
                suma[k] = suma.get(k, 0.0) + v
                conteo[k] = conteo.get(k, 0) + 1
    return {k: suma[k] / conteo[k] for k in suma}

promedio = calcular_reporte_promedio(reportes)

print('=' * 60)
print('📊 RESUMEN PROMEDIO DEL DESAFÍO INTEGRADOR (30 PUZZLES)')
print('=' * 60)
for metrica, val in promedio.items():
    if 'precision' in metrica or 'top1' in metrica:
        print(f'  - {metrica:<30}: {val * 100.0:6.2f}%')
    else:
        print(f'  - {metrica:<30}: {val:8.4f}')
print('=' * 60)


---
## 👁️ 5. Visualización de Reconstrucción (Opcional)

Pueden inspeccionar visualmente cualquier rompecabezas resuelto por su pipeline indicando su índice (0 a 29).


In [ ]:
INDICE_A_INSPECCIONAR = 0

puzzle_ejemplo = puzzles_desafio[INDICE_A_INSPECCIONAR]
print(f'Rompecabezas {INDICE_A_INSPECCIONAR + 1}: {puzzle_ejemplo.metadatos.get("archivo_origen", "puzzle")}')
mostrar_piezas_desordenadas(puzzle_ejemplo.piezas, puzzle_ejemplo.cantidad_filas, puzzle_ejemplo.cantidad_columnas)
